# Preverba: koliko AUC modela TRIM razloži že velikost klona pred zdravljenjem

Zvezek ne trenira ničesar, ne spreminja podatkov in ne potrebuje GPU. Bere `data_labels.pkl`, `df_all_tcrs.pkl` in zapise zagonov v `results/logs` (`x_true`, `y_score`). Za izpuščena bolnika 23 in 24 izračuna:

1. AUC izhodišč iz prave velikosti klona pred zdravljenjem (kri, tumor),
2. AUC modela TRIM znotraj skupin klonov enake velikosti,
3. ali ocena TRIM izboljša razvrstitev, če velikost klona že poznamo (velikost klona, vezi razbije ocena TRIM),
4. AUC na ravni klonov (vsak klon šteje enkrat),
5. delež ekspandiranih celic po velikosti klona in globino vzorčenja po bolnikih,
6. AUC pri oznaki iz števil samo tega bolnika in pri oznaki iz deležev namesto surovih števil.

Celice istega klona imajo isto oznako in skoraj isto oceno, zato niso neodvisne. Intervali DeLong to zanemarijo, zato so dodani intervali bootstrap po klonih (vzorčimo klone s ponavljanjem, vsak izbrani klon prinese vse svoje celice).

**Kako poženeš:** Runtime → Run all (okoli 1 minuto, zadostuje CPU). Rezultati gredo v novo mapo `MyDrive/Diploma/diploma_statistika/preverba_klonov/<čas>/`, obstoječih datotek zvezek ne spreminja.

In [ ]:
import os, sys, glob, json, pickle, time
import numpy as np
import pandas as pd
from scipy.stats import rankdata, norm, spearmanr

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass  # lokalni zagon: poti iz spremenljivk okolja

try:
    from IPython.display import display
except ImportError:
    display = print

PROC_DIR = os.environ.get('PREVERBA_PROC', '/content/drive/MyDrive/Diploma/data/processed')
LOG_DIR  = os.environ.get('PREVERBA_LOGI', os.path.join(PROC_DIR, 'results', 'logs'))
OUT_DIR  = os.path.join(os.environ.get('PREVERBA_IZHOD', '/content/drive/MyDrive/Diploma/diploma_statistika/preverba_klonov'),
                        time.strftime('%Y%m%d-%H%M%S'))
BOLNIKI   = (23, 24)
N_BOOT    = 2000   # ponovitve bootstrapa po klonih
SEME_BOOT = 0
Z95 = norm.ppf(0.975)

os.makedirs(OUT_DIR, exist_ok=False)
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30); pd.set_option('display.max_rows', 200)
print('pandas', pd.__version__, '| numpy', np.__version__)
print('Podatki:', PROC_DIR)
print('Zapisi zagonov:', LOG_DIR)
print('Izhod:', OUT_DIR)

## 1. Podatki in zapisi zagonov

Celice za vrednotenje so izbrane enako kot v `notebooks/04_expansion_eval.ipynb`: celice izpuščenega bolnika iz krvi pred zdravljenjem. Oznaka je `kri po > kri pred` iz števil v `df_all_tcrs` (vsi bolniki skupaj). Zvezek preveri, da se oznake ujemajo z `x_true` v vsakem zapisu, sicer se ustavi.

In [ ]:
with open(os.path.join(PROC_DIR, 'data_labels.pkl'), 'rb') as f:
    data_labels = pickle.load(f)
with open(os.path.join(PROC_DIR, 'df_all_tcrs.pkl'), 'rb') as f:
    df_all_tcrs = pickle.load(f)

col_bloodtumor = data_labels.columns.get_loc('Tissue')
col_prepost    = data_labels.columns.get_loc('Treatment Stage')
col_patient    = data_labels.columns.get_loc('Patient')
col_tcr        = data_labels.columns.get_loc('CDR3(Beta1)')
ID_NULL_TCR    = -1

# Enako kot zvezek 04
x_label = data_labels.values
x_label = x_label[x_label[:, col_tcr] != ID_NULL_TCR]
STEVILA = df_all_tcrs.fillna(0).values   # stolpci '1'..'4': kri pred, kri po, tumor pred, tumor po (vsi bolniki skupaj)
print('data_labels', data_labels.shape, '| df_all_tcrs', df_all_tcrs.shape, list(df_all_tcrs.columns))
assert STEVILA.shape[1] == 4, 'df_all_tcrs nima štirih stolpcev s števili'

celice = pd.DataFrame({'tcr': x_label[:, col_tcr].astype(int),
                       'bolnik': x_label[:, col_patient].astype(int),
                       'tkivo': x_label[:, col_bloodtumor].astype(int),
                       'faza': x_label[:, col_prepost].astype(int)})
KVADRANTI = [(0, 0), (0, 1), (1, 0), (1, 1)]   # (tkivo, faza): kri pred, kri po, tumor pred, tumor po


def stevila_bolnika(pid, tcr):
    """Število celic z istim CDR3 v vsakem kvadrantu, samo pri bolniku pid."""
    sub = celice[celice['bolnik'] == pid]
    out = np.zeros((len(tcr), 4))
    for j, (t, s) in enumerate(KVADRANTI):
        vc = sub.loc[(sub['tkivo'] == t) & (sub['faza'] == s), 'tcr'].value_counts()
        out[:, j] = pd.Series(tcr).map(vc).fillna(0).to_numpy()
    return out


def nalozi_zapise(mapa):
    """Zapisi zagonov (log_run v zvezku 04). Pri podvojenem zagonu vzame prvega po času, kot rezultati_napovedi.py."""
    zagoni = {}
    for pot in sorted(glob.glob(os.path.join(mapa, '**', '*.json'), recursive=True)):
        try:
            with open(pot, encoding='utf-8') as f:
                r = json.load(f)
        except (OSError, ValueError):
            continue
        if not isinstance(r, dict) or not {'variant', 'heldout_patient', 'seed', 'x_true', 'y_score'} <= set(r):
            continue
        if r['seed'] is None:
            continue
        k = (r['variant'], int(r['heldout_patient']), int(r['seed']))
        zagoni.setdefault(k, []).append((str(r.get('timestamp', '')), pot, r))
    glavni = {}
    for k in sorted(zagoni):
        v = sorted(zagoni[k], key=lambda t: t[0])
        if len({t[0] for t in v}) > 1:
            print(f'  {k}: {len(v)} zapisi, uporabim prvega ({os.path.basename(v[0][1])})')
        glavni[k] = v[0][2]
    return glavni


zapisi = nalozi_zapise(LOG_DIR)
assert zapisi, f'V {LOG_DIR} ni zapisov zagonov'
print(f'Zagonov: {len(zapisi)}')
for k in sorted(zapisi):
    print(f'  {k[0]:>13}  P{k[1]}  seme {k[2]}  AUC v zapisu {zapisi[k].get("roc_auc")}')

## 2. Statistične funkcije

AUC je verjetnost, da ima ekspandirana celica višjo oceno od neekspandirane (vezi štejejo pol). DeLong po Sun in Xu (2014). Bootstrap po klonih: v vsaki ponovitvi izberemo toliko klonov, kot jih je v vzorcu, s ponavljanjem, in vzamemo vse njihove celice. Ista izbira klonov velja za vse ocene, zato so razlike parne.

In [ ]:
def auc_stolpci(x, M):
    """AUC za vsak stolpec M (Mann-Whitney s povprečnimi rangi)."""
    x = np.asarray(x, bool); M = np.asarray(M, float).reshape(len(x), -1)
    m = int(x.sum()); n = len(x) - m
    if m == 0 or n == 0:
        return np.full(M.shape[1], np.nan)
    r = rankdata(M, axis=0)
    return (r[x].sum(0) - m * (m + 1) / 2) / (m * n)


def delong(x, M):
    """Hitri DeLong: AUC in kovariančna matrika za stolpce M. Potrebuje vsaj 2 pozitivni in 2 negativni celici."""
    x = np.asarray(x, bool); M = np.asarray(M, float).reshape(len(x), -1).T
    poz, neg = M[:, x], M[:, ~x]; m, n = poz.shape[1], neg.shape[1]
    if m < 2 or n < 2:
        k = M.shape[0]
        return auc_stolpci(x, M.T), np.full((k, k), np.nan)
    tx = rankdata(poz, axis=1); ty = rankdata(neg, axis=1); tz = rankdata(np.hstack([poz, neg]), axis=1)
    auc = tz[:, :m].sum(1) / (m * n) - (m + 1) / (2 * n)
    v01 = (tz[:, :m] - tx) / n; v10 = 1 - (tz[:, m:] - ty) / m
    S = np.atleast_2d(np.cov(v01)) / m + np.atleast_2d(np.cov(v10)) / n
    return auc, S


def boot_auc(x, M, kloni, n_boot=N_BOOT, seme=SEME_BOOT):
    """AUC stolpcev M v n_boot vzorcih klonov s ponavljanjem."""
    x = np.asarray(x, bool); M = np.asarray(M, float).reshape(len(x), -1)
    _, inv = np.unique(kloni, return_inverse=True)
    inv = inv.ravel()
    red = np.argsort(inv, kind='stable'); dolz = np.bincount(inv)
    zac = np.concatenate([[0], np.cumsum(dolz)[:-1]]); K = len(dolz)
    rng = np.random.default_rng(seme)
    out = np.empty((n_boot, M.shape[1]))
    for b in range(n_boot):
        s = rng.integers(0, K, K); L = dolz[s]
        idx = red[np.repeat(zac[s] - np.cumsum(L) + L, L) + np.arange(L.sum())]
        out[b] = auc_stolpci(x[idx], M[idx])
    return out


def interval(v):
    v = v[np.isfinite(v)]
    return (np.nan, np.nan) if len(v) == 0 else tuple(np.percentile(v, [2.5, 97.5]))


def tabela_auc(bolnik, raven, skupina, x, M, imena, kloni, z_bootom=True):
    """Vrstice z AUC, intervali DeLong in bootstrap; vrne še AUC, kovarianco in bootstrap za parne primerjave."""
    auc, S = delong(x, M)
    B = boot_auc(x, M, kloni) if z_bootom else None
    vrstice = []
    for i, ime in enumerate(imena):
        se = np.sqrt(S[i, i]) if np.isfinite(S[i, i]) else np.nan
        bl, bh = interval(B[:, i]) if B is not None else (np.nan, np.nan)
        vrstice.append({'bolnik': bolnik, 'raven': raven, 'skupina': skupina, 'ocena': ime,
                        'n': int(len(x)), 'n_poz': int(np.sum(x)), 'n_klonov': int(len(np.unique(kloni))),
                        'n_poz_klonov': int(len(np.unique(np.asarray(kloni)[np.asarray(x, bool)]))),
                        'auc': auc[i], 'delong_sp': max(0.0, auc[i] - Z95 * se), 'delong_zg': min(1.0, auc[i] + Z95 * se),
                        'boot_sp': bl, 'boot_zg': bh})
    return vrstice, auc, S, B


def primerjava(bolnik, skupina, ime_a, ime_b, i, j, auc, S, B):
    """Parna primerjava AUC ocen i in j: DeLong in (če je B podan) bootstrap po klonih."""
    d = auc[i] - auc[j]
    var = S[i, i] + S[j, j] - 2 * S[i, j]
    z = d / np.sqrt(var) if np.isfinite(var) and var > 0 else np.nan
    db = (B[:, i] - B[:, j]) if B is not None else np.array([])
    db = db[np.isfinite(db)]
    bl, bh = interval(db)
    pb = min(1.0, 2 * min((db <= 0).mean(), (db >= 0).mean())) if len(db) else np.nan
    return {'bolnik': bolnik, 'skupina': skupina, 'ocena_a': ime_a, 'ocena_b': ime_b,
            'auc_a': auc[i], 'auc_b': auc[j], 'razlika': d,
            'delong_p': 2 * norm.sf(abs(z)) if np.isfinite(z) else np.nan,
            'boot_sp': bl, 'boot_zg': bh, 'boot_p': pb}

## 3. Analiza za bolnika 23 in 24

Izhodišča (vsa iz podatkov pred zdravljenjem, ki jih ima tudi TRIM):

| Ocena | Pomen |
|---|---|
| `krv_pred` | število celic klona v krvi pred zdravljenjem, vsi bolniki skupaj (isto število kot v oznaki) |
| `krv_pred_bolnik` | isto, samo celice tega bolnika |
| `krv_pred_ge2` | 1, če ima klon v krvi pred zdravljenjem vsaj 2 celici |
| `tumor_pred` | število celic klona v tumorju pred zdravljenjem |
| `pred_skupaj` | kri in tumor pred zdravljenjem skupaj |

Ocene TRIM: posamezni zagoni (`različica:sN`), povprečje rangov po semenih (`različica:ansambel`), povprečje rangov vseh zagonov (`vse:ansambel`) in delež zagonov z y > 0 (`vse:delez_y>0`). `leks[...]` razvrsti celice po `krv_pred`, vezi razbije ocena TRIM: če ima višji AUC od `krv_pred`, TRIM pove kaj več o klonih enake velikosti.

In [ ]:
IZHODISCA = ['krv_pred', 'krv_pred_bolnik', 'krv_pred_ge2', 'tumor_pred', 'pred_skupaj']
VELIKOSTI = [(1, 1, '1'), (2, 2, '2'), (3, 5, '3-5'), (6, 10, '6-10'), (11, np.inf, '>10')]

v_auc, v_prim, v_vel, v_uje, v_ozn, povzetek = [], [], [], [], [], {}
for pid in BOLNIKI:
    ho = x_label[x_label[:, col_patient] == pid]
    bb = (ho[:, col_bloodtumor] == 0) & (ho[:, col_prepost] == 0)
    tcr = ho[bb, col_tcr].astype(int)
    real_clon = np.take(STEVILA, tcr, axis=0)
    x = real_clon[:, 0] < real_clon[:, 1]
    pre = real_clon[:, 0]

    kljuci = sorted(k for k in zapisi if k[1] == pid)
    if not kljuci:
        print(f'P{pid}: ni zapisov, preskočim'); continue
    for k in kljuci:
        xt = np.asarray(zapisi[k]['x_true'], int)
        if len(xt) != len(x) or not np.array_equal(xt, x.astype(int)):
            raise ValueError(f'P{pid} {k}: oznake iz podatkov se ne ujemajo z x_true v zapisu (n = {len(x)} proti {len(xt)}). '
                             'Podatki so se po zagonu spremenili, primerjava po celicah ni mogoča.')
        a = auc_stolpci(x, np.asarray(zapisi[k]['y_score'], float))[0]
        if zapisi[k].get('roc_auc') is not None and abs(a - zapisi[k]['roc_auc']) > 1e-3:
            print(f'  OPOZORILO {k}: AUC {a:.4f}, v zapisu {zapisi[k]["roc_auc"]}')

    # --- ocene ---
    pac = stevila_bolnika(pid, tcr)
    ocene = {'krv_pred': pre, 'krv_pred_bolnik': pac[:, 0], 'krv_pred_ge2': (pre >= 2).astype(float),
             'tumor_pred': real_clon[:, 2], 'pred_skupaj': pre + real_clon[:, 2]}
    trim = {f'{k[0]}:s{k[2]}': np.asarray(zapisi[k]['y_score'], float) for k in kljuci}
    for razl in sorted({k[0] for k in kljuci}):
        kr = [k for k in kljuci if k[0] == razl]
        if len(kr) > 1:
            trim[f'{razl}:ansambel'] = np.mean([rankdata(zapisi[k]['y_score']) for k in kr], 0)
    trim['vse:ansambel'] = np.mean([rankdata(zapisi[k]['y_score']) for k in kljuci], 0)
    trim['vse:delez_y>0'] = np.mean([np.asarray(zapisi[k]['y_score']) > 0 for k in kljuci], 0)
    leks = {f'leks[{ime}]': pre + rankdata(y) / (len(y) + 1) for ime, y in trim.items()}
    vse = {**ocene, **trim, **leks}
    imena = list(vse); M = np.column_stack([vse[i] for i in imena]); I = {ime: i for i, ime in enumerate(imena)}

    # --- celotna množica in parne primerjave s krv_pred ---
    vr, auc, S, B = tabela_auc(pid, 'celice', 'vse', x, M, imena, tcr)
    v_auc += vr
    for ime in imena:
        if ime != 'krv_pred':
            v_prim.append(primerjava(pid, 'vse', ime, 'krv_pred', I[ime], I['krv_pred'], auc, S, B))

    # --- skupine klonov enake velikosti ---
    skupine = {'pred=1': pre == 1, 'pred=2': pre == 2, 'pred>=3': pre >= 3, 'pred>=2': pre >= 2}
    imena_sk = ['krv_pred', 'krv_pred_bolnik', 'tumor_pred'] + list(trim)
    M_sk = np.column_stack([vse[i] for i in imena_sk])
    for sk, m in skupine.items():
        if m.sum() == 0:
            continue
        vr_s, auc_s, S_s, B_s = tabela_auc(pid, 'celice', sk, x[m], M_sk[m], imena_sk, tcr[m])
        v_auc += vr_s
        if sk == 'pred>=2':
            for ime in trim:
                v_prim.append(primerjava(pid, sk, ime, 'krv_pred', imena_sk.index(ime), 0, auc_s, S_s, B_s))

    # --- raven klonov: vsak klon šteje enkrat (povprečje ocene čez njegove celice) ---
    uk, inv = np.unique(tcr, return_inverse=True); inv = inv.ravel()
    xk = np.bincount(inv, weights=x) / np.bincount(inv)
    assert np.all((xk == 0) | (xk == 1)), 'celice istega klona imajo različne oznake'
    Mk = np.column_stack([np.bincount(inv, weights=M[:, i]) / np.bincount(inv) for i in range(M.shape[1])])
    vr_k, auc_k, S_k, _ = tabela_auc(pid, 'kloni', 'vse', xk.astype(bool), Mk, imena, uk, z_bootom=False)
    v_auc += vr_k
    for ime in trim:   # kloni so neodvisni, zato je tu DeLong primeren
        v_prim.append(primerjava(pid, 'kloni', ime, 'krv_pred', I[ime], I['krv_pred'], auc_k, S_k, None))

    # --- ujemanje psevdoklonov TRIM s pravimi kloni ---
    for ime, y in trim.items():
        poz = y > 0 if ':s' in ime else None
        v_uje.append({'bolnik': pid, 'ocena': ime,
                      'delez_y_poz_pred1': float(poz[pre == 1].mean()) if poz is not None and (pre == 1).any() else np.nan,
                      'delez_y_poz_pred2plus': float(poz[pre >= 2].mean()) if poz is not None and (pre >= 2).any() else np.nan,
                      'spearman_z_krv_pred': spearmanr(y, pre)[0]})

    # --- delež ekspandiranih po velikosti klona ---
    y_poz_kdaj = np.any([np.asarray(zapisi[k]['y_score']) > 0 for k in kljuci], 0)
    for lo, hi, ime in VELIKOSTI:
        m = (pre >= lo) & (pre <= hi)
        if m.any():
            km = np.unique(tcr[m]); kx = np.unique(tcr[m & x])
            v_vel.append({'bolnik': pid, 'velikost_krv_pred': ime, 'n_celic': int(m.sum()), 'n_klonov': len(km),
                          'n_eksp_celic': int(x[m].sum()), 'delez_eksp_celic': float(x[m].mean()),
                          'n_eksp_klonov': len(kx), 'delez_eksp_klonov': len(kx) / len(km),
                          'delez_y_poz_vsaj_enkrat': float(y_poz_kdaj[m].mean())})

    # --- druge oznake ---
    n_pre = int(((celice['bolnik'] == pid) & (celice['tkivo'] == 0) & (celice['faza'] == 0)).sum())
    n_post = int(((celice['bolnik'] == pid) & (celice['tkivo'] == 0) & (celice['faza'] == 1)).sum())
    oznake = {'stevila_vsi_bolniki': x, 'stevila_bolnik': pac[:, 1] > pac[:, 0]}
    if n_post > 0:
        oznake['delezi_bolnik'] = pac[:, 1] / n_post > pac[:, 0] / n_pre
    for ime_o, xo in oznake.items():
        a_o = auc_stolpci(xo, M)
        for i, ime in enumerate(imena):
            if not ime.startswith('leks['):
                v_ozn.append({'bolnik': pid, 'oznaka': ime_o, 'n_poz': int(xo.sum()),
                              'n_drugacnih_od_glavne': int((xo != x).sum()), 'ocena': ime, 'auc': a_o[i]})

    # --- povzetek ---
    def vrstica(sk, ime, raven='celice'):
        return next((r for r in v_auc if r['bolnik'] == pid and r['raven'] == raven and r['skupina'] == sk and r['ocena'] == ime), None)
    def prim(sk, ime):
        return next((r for r in v_prim if r['bolnik'] == pid and r['skupina'] == sk and r['ocena_a'] == ime), None)
    glavne = [i for i in trim if i.endswith(':ansambel')]
    povzetek[f'P{pid}'] = {
        'n_celic': int(len(x)), 'n_eksp_celic': int(x.sum()), 'n_klonov': int(len(uk)), 'n_eksp_klonov': int(xk.sum()),
        'celice_krv_pred_po': [n_pre, n_post],
        'auc_krv_pred': vrstica('vse', 'krv_pred'),
        'auc_trim': {i: vrstica('vse', i) for i in glavne},
        'trim_minus_krv_pred': {i: prim('vse', i) for i in glavne},
        'leks_minus_krv_pred': {i: prim('vse', f'leks[{i}]') for i in glavne},
        'auc_trim_v_skupinah': {sk: {i: vrstica(sk, i) for i in ['krv_pred'] + glavne} for sk in skupine},
        'auc_klonov': {i: vrstica('vse', i, 'kloni') for i in ['krv_pred'] + glavne},
        'trim_minus_krv_pred_kloni': {i: prim('kloni', i) for i in glavne},
    }
    print(f'P{pid}: {len(x)} celic ({int(x.sum())} ekspandiranih), {len(uk)} klonov ({int(xk.sum())} ekspandiranih), '
          f'kri pred/po {n_pre}/{n_post} celic, zagonov {len(kljuci)}')

df_auc, df_prim = pd.DataFrame(v_auc), pd.DataFrame(v_prim)
df_vel, df_uje, df_ozn = pd.DataFrame(v_vel), pd.DataFrame(v_uje), pd.DataFrame(v_ozn)

## 4. Rezultati

**Glavna tabela:** AUC izhodišč in ocen TRIM na vseh celicah. `boot_*` je 95-% interval bootstrap po klonih, `delong_*` interval DeLong (preozek, ker celice istega klona niso neodvisne).

In [ ]:
def pokazi(df, stolpci=None, decimalke=3):
    d = df if stolpci is None else df[stolpci]
    display(d.round(decimalke).reset_index(drop=True))

STOLPCI = ['bolnik', 'ocena', 'n', 'n_poz', 'n_klonov', 'n_poz_klonov', 'auc', 'boot_sp', 'boot_zg', 'delong_sp', 'delong_zg']
glavna = df_auc[(df_auc['raven'] == 'celice') & (df_auc['skupina'] == 'vse') & ~df_auc['ocena'].str.startswith('leks[')]
pokazi(glavna, STOLPCI)

**TRIM proti velikosti klona:** razlika AUC (ocena − `krv_pred`). Če interval bootstrap vključuje 0, TRIM na tej oznaki ni boljši od samega števila celic klona pred zdravljenjem. Vrstice `leks[...]` povedo, ali ocena TRIM pomaga razvrstiti klone enake velikosti.

In [ ]:
pokazi(df_prim[df_prim['skupina'] == 'vse'], ['bolnik', 'ocena_a', 'auc_a', 'auc_b', 'razlika', 'boot_sp', 'boot_zg', 'boot_p', 'delong_p'])

**AUC znotraj klonov enake velikosti** (`pred=1`: klon ima v krvi pred zdravljenjem eno celico; ekspandiran je, če ima po zdravljenju vsaj dve). Pri malo pozitivnih klonih (`n_poz_klonov`) so ocene nezanesljive.

In [ ]:
sk = df_auc[(df_auc['raven'] == 'celice') & (df_auc['skupina'] != 'vse')]
pokazi(sk.sort_values(['bolnik', 'skupina', 'ocena']), ['bolnik', 'skupina', 'ocena', 'n', 'n_poz', 'n_klonov', 'n_poz_klonov', 'auc', 'boot_sp', 'boot_zg'])
pokazi(df_prim[df_prim['skupina'] == 'pred>=2'], ['bolnik', 'skupina', 'ocena_a', 'auc_a', 'auc_b', 'razlika', 'boot_sp', 'boot_zg', 'boot_p'])

**Raven klonov** (vsak klon šteje enkrat, ocena je povprečje čez njegove celice; interval DeLong, kloni so neodvisni). Druga tabela primerja ocene TRIM s `krv_pred` na ravni klonov (parni DeLong).

In [ ]:
pokazi(df_auc[df_auc['raven'] == 'kloni'], ['bolnik', 'ocena', 'n', 'n_poz', 'auc', 'delong_sp', 'delong_zg'])
pokazi(df_prim[df_prim['skupina'] == 'kloni'], ['bolnik', 'ocena_a', 'auc_a', 'auc_b', 'razlika', 'delong_p'])

**Ali psevdokloni TRIM sledijo pravim klonom:** delež celic z y > 0 med celicami iz klonov z eno celico in med celicami iz klonov z vsaj dvema celicama pred zdravljenjem, ter Spearmanova korelacija ocene z velikostjo klona.

In [ ]:
pokazi(df_uje)

**Delež ekspandiranih po velikosti klona v krvi pred zdravljenjem.**

In [ ]:
pokazi(df_vel)

## 5. Oznaka in globina vzorčenja

Oznaka primerja surova števila celic, zato je odvisna od tega, koliko celic je bilo zajetih pred zdravljenjem in po njem. Tabela kaže za vsakega bolnika število celic v vsakem kvadrantu, število klonov v krvi pred zdravljenjem, koliko jih je ekspandiranih (vseh in tistih z eno celico pred zdravljenjem) ter delež ekspandiranih celic krvi pred zdravljenjem pri treh oznakah: števila vseh bolnikov skupaj (kot TRIM), števila samo tega bolnika in deleži (število celic klona deljeno s številom vseh celic bolnika ob tem času).

In [ ]:
v_glob = []
for pid_ in sorted(celice['bolnik'].unique()):
    sub = celice[celice['bolnik'] == pid_]
    n = {kv: int(((sub['tkivo'] == kv[0]) & (sub['faza'] == kv[1])).sum()) for kv in KVADRANTI}
    vr = {'bolnik': int(pid_), 'krv_pred': n[(0, 0)], 'krv_po': n[(0, 1)], 'tumor_pred': n[(1, 0)], 'tumor_po': n[(1, 1)]}
    if n[(0, 0)] and n[(0, 1)]:
        t = sub.loc[(sub['tkivo'] == 0) & (sub['faza'] == 0), 'tcr'].to_numpy()
        rc, pc = STEVILA[t], stevila_bolnika(pid_, t)
        uk_, inv_ = np.unique(t, return_inverse=True); inv_ = inv_.ravel()
        eksp_k = np.bincount(inv_, weights=rc[:, 0] < rc[:, 1]) > 0
        vr.update({'klonov': len(uk_), 'eksp_klonov': int(eksp_k.sum()),
                   'eksp_klonov_z_1_celico': int((eksp_k & (STEVILA[uk_, 0] == 1)).sum()),
                   'razmerje_po_pred': n[(0, 1)] / n[(0, 0)],
                   'delez_eksp_vsi_bolniki': float((rc[:, 0] < rc[:, 1]).mean()),
                   'delez_eksp_bolnik': float((pc[:, 0] < pc[:, 1]).mean()),
                   'delez_eksp_delezi': float((pc[:, 1] / n[(0, 1)] > pc[:, 0] / n[(0, 0)]).mean()),
                   'delez_celic_v_klonih_z_1_celico': float((pc[:, 0] == 1).mean())})
    v_glob.append(vr)
df_glob = pd.DataFrame(v_glob)
pokazi(df_glob)
ok = df_glob.dropna(subset=['razmerje_po_pred'])
korelacije = {}
if len(ok) >= 4:
    for st in ['delez_eksp_vsi_bolniki', 'delez_eksp_bolnik', 'delez_eksp_delezi']:
        r, p = spearmanr(ok['razmerje_po_pred'], ok[st])
        korelacije[st] = {'spearman': float(r), 'p': float(p), 'n_bolnikov': int(len(ok))}
        print(f'Spearman(razmerje celic po/pred, {st}) = {r:.2f} (p = {p:.3f}, n = {len(ok)} bolnikov)')

**AUC pri drugih oznakah** (iste celice, iste ocene). `stevila_bolnik`: kri po > kri pred samo iz celic tega bolnika. `delezi_bolnik`: delež klona v krvi po zdravljenju > delež pred zdravljenjem.

In [ ]:
tab_ozn = df_ozn.pivot_table(index=['bolnik', 'ocena'], columns='oznaka', values='auc', sort=False)
display(tab_ozn.round(3))
display(df_ozn.groupby(['bolnik', 'oznaka'], sort=False)[['n_poz', 'n_drugacnih_od_glavne']].first())

## 6. Shranjevanje

In [ ]:
def v_json(o):
    if isinstance(o, dict): return {str(k): v_json(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [v_json(v) for v in o]
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating, float)): return None if not np.isfinite(o) else round(float(o), 4)
    if isinstance(o, np.bool_): return bool(o)
    return o

df_auc.to_csv(os.path.join(OUT_DIR, 'auc.csv'), index=False)
df_prim.to_csv(os.path.join(OUT_DIR, 'primerjave.csv'), index=False)
df_vel.to_csv(os.path.join(OUT_DIR, 'velikosti_klonov.csv'), index=False)
df_uje.to_csv(os.path.join(OUT_DIR, 'ujemanje_psevdoklonov.csv'), index=False)
df_ozn.to_csv(os.path.join(OUT_DIR, 'druge_oznake.csv'), index=False)
df_glob.to_csv(os.path.join(OUT_DIR, 'globina_po_bolnikih.csv'), index=False)
with open(os.path.join(OUT_DIR, 'povzetek.json'), 'w', encoding='utf-8') as f:
    json.dump(v_json({'bolniki': povzetek, 'globina_korelacije': korelacije, 'n_boot': N_BOOT,
                      'zagoni': [f'{k[0]}:P{k[1]}:s{k[2]}' for k in sorted(zapisi)],
                      'verzije': {'pandas': pd.__version__, 'numpy': np.__version__}}), f, indent=2, ensure_ascii=False)
print('Shranjeno v', OUT_DIR)
for ime in sorted(os.listdir(OUT_DIR)):
    print(' ', ime)

## Kako brati

- `krv_pred` blizu AUC modela TRIM (interval razlike vključuje 0): AUC na tej oznaki večinoma pride iz velikosti klona pred zdravljenjem. Različice vhodov (RNA, fluksi, filter genov) potem primerjamo znotraj skupin klonov enake velikosti (predvsem `pred=1`) ali na ravni klonov, ne po skupnem AUC.
- TRIM nad `krv_pred` v skupini `pred=1`, na ravni klonov ali pri `leks[...]`: TRIM pove nekaj več o klonih enake velikosti, zato je smiselno preveriti, ali to pride iz metabolnih genov (filter genov iz csGEM).
- `n_poz_klonov` pove, na koliko ekspandiranih klonih sloni ocena. Pri nekaj klonih so razlike med različicami nezanesljive ne glede na število celic.
- Delež ekspandiranih, ki raste z razmerjem celic po/pred med bolniki, pri oznaki iz deležev pa ne, pomeni, da surova števila v oznaki merijo tudi globino vzorčenja.